In [1]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

c:\Users\jarriaza\sansebastian-bot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\jarriaza\AppData\Local\Temp\ipykernel_16416\941289217.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [5]:
model = genai.GenerativeModel("gemini-3.1-flash-lite")

response = model.generate_content("Recomiéndame en una frase qué visitar en San Sebastián")

print(response.text)

No puedes perderte un paseo por la bahía de la Concha para terminar degustando los famosos "pintxos" en el encanto histórico de la Parte Vieja.


In [6]:
with open("docs/parte_vieja.md", "r", encoding="utf-8") as f:
    texto = f.read()

# Cortamos en cada "## " — cada corte es un monumento (o la intro, o fuentes)
secciones = texto.split("\n## ")
secciones = [s if s.startswith("#") else "## " + s for s in secciones]

# Excluimos la sección de Fuentes: son citas, no contenido que el bot deba usar para responder
chunks = [s.strip() for s in secciones if "## Fuentes" not in s]

print(f"Total de chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"--- Chunk {i}: {c[:60]}...")

Total de chunks: 9
--- Chunk 0: # La Parte Vieja de San Sebastián (Donostia)

La Parte Vieja...
--- Chunk 1: ## Basílica de Santa María del Coro

La basílica de Santa Ma...
--- Chunk 2: ## Iglesia de San Vicente Mártir

La iglesia de San Vicente ...
--- Chunk 3: ## Plaza de la Constitución

La Plaza de la Constitución, co...
--- Chunk 4: ## Museo San Telmo

El Museo San Telmo, situado en la plaza ...
--- Chunk 5: ## Alameda del Boulevard

La Alameda del Boulevard, conocida...
--- Chunk 6: ## Calle 31 de Agosto

La calle 31 de Agosto es la única cal...
--- Chunk 7: ## Ayuntamiento de San Sebastián (antiguo Gran Casino)

El a...
--- Chunk 8: ## Monte Urgull (conjunto asociado a la Parte Vieja)

El mon...


In [8]:
resultado = genai.embed_content(
    model="models/gemini-embedding-001",
    content=chunks,
    task_type="retrieval_document",
    output_dimensionality=768
)

embeddings = resultado["embedding"]
print(f"Total de embeddings generados: {len(embeddings)}")
print(f"Dimensiones de cada vector: {len(embeddings[0])}")

Total de embeddings generados: 9
Dimensiones de cada vector: 768


In [10]:
import numpy as np

# Guardamos chunk + su vector juntos, para no perder la asociación
base_vectorial = list(zip(chunks, embeddings))

def buscar_relevantes(pregunta, top_n=2):
    # Convertimos la PREGUNTA a vector — nota el task_type distinto
    resultado_pregunta = genai.embed_content(
        model="models/gemini-embedding-001",
        content=pregunta,
        task_type="retrieval_query",
        output_dimensionality=768
    )
    vector_pregunta = resultado_pregunta["embedding"]

    # Comparamos ese vector contra los 9 vectores guardados
    similitudes = []
    for chunk_texto, chunk_vector in base_vectorial:
        similitud = np.dot(vector_pregunta, chunk_vector)
        similitudes.append((similitud, chunk_texto))

    # Ordenamos de más similar a menos, devolvemos los top_n
    similitudes.sort(key=lambda x: x[0], reverse=True)
    return [texto for _, texto in similitudes[:top_n]]

In [14]:
resultados = buscar_relevantes("¿cuándo se construyó la Basílica de Santa María?")
for r in resultados:
    print(r[:100], "\n---")

## Basílica de Santa María del Coro

La basílica de Santa María del Coro se encuentra en el extremo  
---
## Iglesia de San Vicente Mártir

La iglesia de San Vicente Mártir, en la calle San Juan, es conside 
---


In [15]:
def responder(pregunta, top_n=2):
    # Fase de consulta: buscamos los chunks relevantes (lo que ya probamos)
    chunks_relevantes = buscar_relevantes(pregunta, top_n=top_n)
    contexto = "\n\n---\n\n".join(chunks_relevantes)

    modelo_chat = genai.GenerativeModel(
        "gemini-3.1-flash-lite",
        system_instruction=(
            "Eres un guía turístico de San Sebastián. Responde SOLO con base en el "
            "contexto proporcionado. Si el contexto no tiene la respuesta, dilo "
            "claramente en vez de inventar información."
        )
    )

    prompt = f"Contexto:\n{contexto}\n\nPregunta del usuario: {pregunta}"
    respuesta = modelo_chat.generate_content(prompt)
    return respuesta.text

In [16]:
chunks_relevantes = buscar_relevantes("¿cuándo se construyó la Basílica de Santa María?")
contexto = "\n\n---\n\n".join(chunks_relevantes)

modelo_chat = genai.GenerativeModel(
    "gemini-3.1-flash-lite",
    system_instruction="Eres un guía turístico de San Sebastián. Responde SOLO con base en el contexto proporcionado."
)

prompt = f"Contexto:\n{contexto}\n\nPregunta del usuario: ¿cuándo se construyó la Basílica de Santa María?"
respuesta = modelo_chat.generate_content(prompt)

print(respuesta)

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "La construcci\u00f3n de la actual bas\u00edlica, de estilo barroco, comenz\u00f3 en 1743 y concluy\u00f3 en 1774.\n\nCabe precisar que en ese mismo emplazamiento existi\u00f3 una primera iglesia rom\u00e1nica entre los siglos XII y XIII, la cual fue ampliada en estilo g\u00f3tico-renacentista entre 1522 y 1560. Tras sufrir da\u00f1os graves en 1688, se realiz\u00f3 la reconstrucci\u00f3n barroca que hoy se conserva."
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 923,
        "candidates_token_count": 104,
        "total_token_count": 1027
      },
      "model_version": "gemini-3.1-flash-lite"
    }),
)


In [ ]:
print(responder("¿Qué monumentos son imprecindibles en la parte vieja?"))

En la Parte Vieja, los elementos de mayor relevancia patrimonial son las dos parroquias mayores, los restos de la muralla y el antiguo casco amurallado. Además, la calle 31 de Agosto es un lugar destacado por ser la única que sobrevivió al incendio de 1813.
